In [17]:
import random
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np
import os

# preprocess.ipynb文件的作用是生成train.csv和test.csv
##### 比例为80%：20%，即3320个user：831个user的记录
##### 字段为：'user_id', 'q_idx', 's_idx', 'q_type', 'q_diff', 'ms_first_response', 'attempt_count', 'correct'

In [18]:
seed = 42
np.random.seed(seed)

In [19]:
df = pd.read_csv("assist09_processed.csv", low_memory=False, encoding="ISO-8859-1")
df.columns,df.shape

(Index(['Unnamed: 0', 'order_id', 'assignment_id', 'user_id', 'assistment_id',
        'problem_id', 'original', 'correct', 'attempt_count',
        'ms_first_response', 'tutor_mode', 'answer_type', 'sequence_id',
        'student_class_id', 'position', 'type', 'base_sequence_id', 'skill_id',
        'skill_name', 'teacher_id', 'school_id', 'hint_count', 'hint_total',
        'overlap_time', 'template_id', 'answer_id', 'answer_text',
        'first_action', 'bottom_hint', 'opportunity', 'opportunity_original'],
       dtype='object'),
 (274590, 31))

In [20]:
key = 'problem_id'

key_q = 'q_idx'
key_s = 's_idx'
key_qt = 'q_type'
key_qd = 'q_diff'
key_ms_first_response = 'ms_first_response'
key_attempts = 'attempt_count'

question_id_dict = dict(zip(df[key].unique(), range(len(df[key].unique()))))
skill_id_dict = dict(zip(df['skill_name'].unique(), range(len(df['skill_name'].unique()))))
question_type_dict = dict(zip(df['answer_type'].unique(), range(len(df['answer_type'].unique()))))

# question idx
df[key_q] = df[key].map(question_id_dict)
df[key_s] = df['skill_name'].map(skill_id_dict)
df[key_qt] = df['answer_type'].map(question_type_dict)

n_question = len(question_id_dict)
n_skill = len(skill_id_dict)
n_question_type = len(question_type_dict)

print("num of question:{}, num of skill:{}, n_question_type:{}".format(n_question, n_skill, n_question_type))

num of question:16891, num of skill:101, n_question_type:5


In [21]:
df.columns, df.shape # 多了3列，'q_idx', 's_idx', 'q_type'

(Index(['Unnamed: 0', 'order_id', 'assignment_id', 'user_id', 'assistment_id',
        'problem_id', 'original', 'correct', 'attempt_count',
        'ms_first_response', 'tutor_mode', 'answer_type', 'sequence_id',
        'student_class_id', 'position', 'type', 'base_sequence_id', 'skill_id',
        'skill_name', 'teacher_id', 'school_id', 'hint_count', 'hint_total',
        'overlap_time', 'template_id', 'answer_id', 'answer_text',
        'first_action', 'bottom_hint', 'opportunity', 'opportunity_original',
        'q_idx', 's_idx', 'q_type'],
       dtype='object'),
 (274590, 34))

In [22]:
group1 = df[[key, 'correct']].groupby([key]).apply(lambda r:r['correct'].sum() / len(r['correct']))
group1

problem_id
83        0.5
84        0.0
85        0.2
86        0.5
249       0.0
         ... 
189408    1.0
189554    1.0
189565    0.0
189566    1.0
196456    0.0
Length: 16891, dtype: float64

In [23]:
df[key_qd] = df['problem_id'].map(group1)


In [24]:
df.columns, df.shape # 多了每个问题的难度，q_diff

(Index(['Unnamed: 0', 'order_id', 'assignment_id', 'user_id', 'assistment_id',
        'problem_id', 'original', 'correct', 'attempt_count',
        'ms_first_response', 'tutor_mode', 'answer_type', 'sequence_id',
        'student_class_id', 'position', 'type', 'base_sequence_id', 'skill_id',
        'skill_name', 'teacher_id', 'school_id', 'hint_count', 'hint_total',
        'overlap_time', 'template_id', 'answer_id', 'answer_text',
        'first_action', 'bottom_hint', 'opportunity', 'opportunity_original',
        'q_idx', 's_idx', 'q_type', 'q_diff'],
       dtype='object'),
 (274590, 35))

In [25]:
# 我们需要将数据进行预处理，每个学生的学习记录利用group by合并为序列。
group = df[['user_id', key_q, key_s, key_qt, key_qd, key_ms_first_response, key_attempts, 'correct']].groupby(['user_id']).apply(lambda r: (
            r[key_q].values,
            r[key_s].values,
            r[key_qt].values,
            r[key_qd].values,
            r[key_ms_first_response].values,
            r[key_attempts].values,
            r['correct'].values
            ))

group.iloc[0] # 一个user的记录，包含这个user的q_idx, s_idx, q_type, q_diff, ms_first_response, attempt_count, correct

(array([ 132,  133,  134,  135,  136,  137,  138,  139,  140,  141,  142,
         143, 2208, 2209, 2210, 2211, 2377, 2378, 2379]),
 array([ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  9,  9,  9,  9, 10,
        10, 10]),
 array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([0.41666667, 0.37931034, 0.34782609, 0.35135135, 0.25806452,
        0.43478261, 0.41666667, 0.52631579, 0.83333333, 0.77777778,
        0.625     , 0.66666667, 0.58333333, 0.22222222, 0.45454545,
        0.375     , 0.6       , 0.72727273, 0.83333333]),
 array([26271, 29123, 13779, 16901, 11079,  8244, 21884, 59891, 64234,
        82572, 30678, 27225,  9236,  3426, 11915,  2212, 11635, 11012,
        11417]),
 array([1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1]),
 array([0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1]))

### split train test, and generate df

In [26]:
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# train, test = train_test_split(group, test_size=0.2, random_state=seed)
# train, test # 3320, 831, 总共4151个user
train, test = train_test_split(group, test_size=0.4, random_state=seed)
train, test # 

(user_id
 89195    ([5444, 5666, 5497, 5443, 5709, 5705, 5472, 59...
 92303    ([8517], [41], [1], [0.6842105263157895], [104...
 91616    ([12392, 13929, 13935, 14000, 13993, 13939], [...
 80234    ([336, 327, 257, 374, 361, 330, 176, 255, 320,...
 81946    ([5626, 5691, 5479, 5439, 5540, 5595, 5723, 57...
                                ...                        
 89951    ([2917, 3056, 3057, 3105, 3112, 3188, 3189, 31...
 78517    ([2094, 1996, 2077, 8150, 8132, 8102, 12981, 1...
 88159    ([11740], [61], [0], [0.574468085106383], [868...
 91749    ([3, 110, 243], [0, 0, 1], [0, 0, 0], [0.75, 0...
 79460    ([3006, 8305, 8253, 8306, 8341, 8648, 8638, 86...
 Length: 2490, dtype: object,
 user_id
 82061    ([3555, 3718, 12770, 12779, 12769, 12814, 1282...
 96216    ([25, 67, 31, 47, 68, 72, 30, 59, 39, 205, 302...
 87584    ([14354, 14669, 14489, 15100, 14997, 14520, 14...
 81221      ([5996], [30], [0], [0.675], [19833], [1], [1])
 78947    ([138, 249, 247, 258, 231, 196, 168, 135, 

In [27]:
def generate_df_by_group(group):
    df = pd.DataFrame(columns=['user_id', 'q_idx', 's_idx', 'q_type', 'q_diff', 'ms_first_response', 'attempt_count', 'correct'])
    for user_id, (q, s, qt, qd, ms_first_response, attempts, correct) in tqdm(group.items()):
        for i in range(len(q)):
            tmp = [user_id, q[i], s[i], qt[i], qd[i], ms_first_response[i], attempts[i], correct[i]]
            df.loc[len(df)] = tmp
    return df

df_train = generate_df_by_group(train)
df_test = generate_df_by_group(test)

2490it [07:04,  5.86it/s]
1661it [03:19,  8.33it/s]


### mimmax最大最小值缩放sacler.fit_transform():针对字段'ms_first_response', 'attempt_count'做缩放操作
https://zhuanlan.zhihu.com/p/110798359

In [28]:
from sklearn.preprocessing import MinMaxScaler

sacler = MinMaxScaler()

tmp = sacler.fit_transform(df_train[[key_ms_first_response, key_attempts]])
df_train[[key_ms_first_response, key_attempts]] = tmp

tmp = sacler.transform(df_test[[key_ms_first_response, key_attempts]])
df_test[[key_ms_first_response, key_attempts]] = tmp

In [29]:
df_train.columns, df_train.shape

(Index(['user_id', 'q_idx', 's_idx', 'q_type', 'q_diff', 'ms_first_response',
        'attempt_count', 'correct'],
       dtype='object'),
 (169421, 8))

In [30]:
df_train.to_csv("train_6.csv", index=None) # 3320个user的219099条记录

In [31]:
df_test.columns, df_test.shape

(Index(['user_id', 'q_idx', 's_idx', 'q_type', 'q_diff', 'ms_first_response',
        'attempt_count', 'correct'],
       dtype='object'),
 (105169, 8))

In [32]:
df_test.to_csv("test_4.csv", index=None) # 831个user的55491记录